# 00 · Prueba rápida en Colab (antes de las corridas oficiales)

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

Este notebook **no produce resultados para el informe**: verifica en ~10–20 minutos que todo el
sistema funciona en la GPU de Colab antes de lanzar las corridas largas (notebooks 03, 05 y 06).

| Verificación | Qué confirma |
|---|---|
| §1 GPU y versiones | Que la sesión tiene GPU (T4 o mejor) y las librerías correctas |
| §2 Archivos en Drive | Que se subieron el catálogo, las consultas, los índices y las grabaciones |
| §3 Recuperador | BM25 + e5 + re-ranker en GPU |
| §4 LLM en 4-bit | Qué candidatos cargan (Qwen3.5-9B es multimodal: es el de mayor riesgo) y si llaman herramientas |
| §5 ASR y TTS | Whisper (faster-whisper) y XTTS-v2 con la voz del autor, en GPU |
| §6 Punta a punta | El escenario de voz **V1** completo |
| §7 Resumen | Memoria máxima de GPU: ¿caben LLM + ASR + TTS + recuperador juntos? |

**Antes de ejecutar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*. Luego
*Entorno de ejecución → Ejecutar todas*. Si la instalación de §0 pide reiniciar la sesión,
reinicia y vuelve a ejecutar todo.

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = "rag,llm,voz"
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

## 1. GPU y versiones

In [2]:
import importlib.metadata as md
import time

import asistente_compras  # noqa: F401  (fijar JAX en CPU y el asignador de memoria de la GPU)
import torch

assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} · {gpu.total_memory / 2**30:.1f} GB")
for pkg in ("torch", "transformers", "bitsandbytes", "accelerate", "sentence-transformers",
            "faster-whisper", "ctranslate2", "coqui-tts", "speechbrain"):
    try:
        print(f"  {pkg:22s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:22s} ✗ NO INSTALADO")


def vram(etapa: str, registro: dict = {}) -> None:
    """Memoria de GPU usada ahora y máxima desde el inicio (GB)."""
    torch.cuda.synchronize()
    ahora, pico = torch.cuda.memory_allocated() / 2**30, torch.cuda.max_memory_allocated() / 2**30
    registro[etapa] = round(pico, 2)
    print(f"   VRAM tras {etapa}: {ahora:.2f} GB (pico {pico:.2f} GB de {gpu.total_memory / 2**30:.1f})")


RESUMEN = {}

GPU: NVIDIA L4 · 22.0 GB
  torch                  2.14.1+cu130
  transformers           5.22.1
  bitsandbytes           0.50.2
  accelerate             1.15.0
  sentence-transformers  5.7.0
  faster-whisper         1.2.1
  ctranslate2            4.9.0
  coqui-tts              0.28.0
  speechbrain            1.1.2


## 2. Archivos en Drive

In [3]:
from asistente_compras.config import get_paths
from asistente_compras.scenarios import SCENARIOS

paths = get_paths().ensure()
VOICE = [s for s in SCENARIOS if s.voice]
requeridos = {
    "catálogo": paths.processed / "catalog.parquet",
    "consultas (test)": paths.processed / "queries_test.parquet",
    "qrels (test)": paths.eval / "qrels_test.parquet",
    "voz de referencia": paths.root / "voz" / "referencia.wav",
}
for s in VOICE:
    for i in range(1, len(s.turns) + 1):
        requeridos[f"{s.id} turno {i}"] = paths.root / "voz" / "escenarios" / s.id / f"turno_{i:02d}.wav"
faltan = [k for k, p in requeridos.items() if not p.exists()]
print(f"Datos en: {paths.root}")
print(f"Archivos verificados: {len(requeridos) - len(faltan)}/{len(requeridos)}")
print("Índices e5 precalculados:", [p.name for p in paths.index.glob("*.npy")] or "ninguno (se calculan, ~5 min)")
assert not faltan, f"Faltan: {faltan}"

Archivos verificados: 29/29
Índices e5 precalculados: ['multilingual-e5-base_titulo_marca-w200_229758f91a.npy', 'multilingual-e5-base_titulo_marca_bullets-w200_06dbc3e034.npy']


## 3. Recuperador (BM25 + e5 + re-ranker)

In [4]:
import pandas as pd

from asistente_compras.retrieval import build_system_retriever

catalog = pd.read_parquet(paths.processed / "catalog.parquet")
t = time.perf_counter()
retriever = build_system_retriever(catalog, paths.index)
RESUMEN["carga recuperador (s)"] = round(time.perf_counter() - t, 1)
t = time.perf_counter()
hits = retriever.search("cable usb tipo c", k=3)
print(f"Carga {RESUMEN['carga recuperador (s)']} s · búsqueda {time.perf_counter() - t:.2f} s")
titles = catalog.set_index("product_id").title
for pid, sc in hits:
    print(f"   {sc:6.3f}  {titles[pid][:80]}")
vram("recuperador", RESUMEN)

Carga 31.8 s · búsqueda 1.47 s
   VRAM tras recuperador: 1.58 GB (pico 3.11 GB de 22.0)


## 4. LLM en 4-bit: ¿qué candidatos cargan?

Se prueban en orden de preferencia y se **queda cargado el primero que funcione** (para §6).
Cada uno se descarga de Hugging Face la primera vez (~15–18 GB, 5–10 min). Una prueba "funciona"
si carga y, ante un pedido de compra, **llama a la herramienta `buscar_productos`**.

> Qwen3.5-9B se publica como modelo multimodal (`Qwen3_5ForConditionalGeneration`); el código
> intenta `AutoModelForCausalLM` y, si no aplica, lo carga como modelo imagen-texto usando solo
> texto. Sus capas de atención lineal son más lentas sin los kernels opcionales
> `flash-linear-attention`: por eso se mide el tiempo por respuesta.

In [5]:
# Kernels rápidos para la atención lineal de Qwen3.5 (sin ellos funciona, pero mucho más lento:
# en la primera prueba en L4, ~95 s por turno). Instalarlos solo en GPUs modernas (L4, A100;
# capacidad de cómputo >= 8); en una T4 pueden no ser compatibles.
ACELERAR_QWEN35 = torch.cuda.get_device_capability(0)[0] >= 8
if ACELERAR_QWEN35:
    get_ipython().run_line_magic("pip", "install -q flash-linear-attention")
    import importlib

    importlib.invalidate_caches()
print("Kernels de atención lineal:", "instalados" if ACELERAR_QWEN35 else "no (GPU antigua)")
RESUMEN["kernels atención lineal"] = "sí" if ACELERAR_QWEN35 else "no"
print("Precisión de cómputo del LLM:", "bfloat16" if torch.cuda.is_bf16_supported() else "float16")

Kernels de atención lineal: instalados
Precisión de cómputo del LLM: bfloat16


In [6]:
import traceback

from asistente_compras.agent import SYSTEM_PROMPT, TOOL_SCHEMAS
from asistente_compras.llm import free_gpu, make_llm

CANDIDATOS = ["qwen3.5:9b", "qwen3:8b", "qwen2.5:7b"]
prueba = [{"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": "Necesito dos cables USB tipo C, mi presupuesto es de 40 euros."}]
estado_llm, llm, MODELO = [], None, None
for m in CANDIDATOS:
    print(f"\n── {m}")
    try:
        t = time.perf_counter()
        cand = make_llm(m)
        t_carga = time.perf_counter() - t
        r = cand.chat(prueba, TOOL_SCHEMAS)
        tools = [c["function"]["name"] for c in r.tool_calls]
        ok = "buscar_productos" in tools
        print(f"   cargó en {t_carga:.0f} s · respuesta {r.seconds:.1f} s "
              f"({r.completion_tokens / max(r.seconds, 1e-6):.1f} tokens/s, prompt "
              f"{r.prompt_tokens} tokens) · herramientas {tools}")
        if r.content:
            print(f"   texto: {r.content[:200]!r}")
        RESUMEN[f"LLM {m}: s por llamada"] = round(r.seconds, 1)
        vram(f"LLM {m}", RESUMEN)
        estado_llm.append({"modelo": m, "carga": "✅", "herramientas": "✅" if ok else "❌",
                           "carga_s": round(t_carga), "respuesta_s": round(r.seconds, 1)})
        if ok:
            llm, MODELO = cand, m
            break
        del cand
        free_gpu()
    except Exception as e:  # registrar el error y probar el siguiente candidato
        traceback.print_exc(limit=2)
        estado_llm.append({"modelo": m, "carga": f"❌ {type(e).__name__}: {str(e)[:80]}",
                           "herramientas": "—", "carga_s": None, "respuesta_s": None})
        free_gpu()
display(pd.DataFrame(estado_llm).set_index("modelo"))
assert llm is not None, "Ningún candidato funcionó: revisar los errores de arriba"
print(f"\nModelo para la prueba punta a punta: {MODELO}")


── qwen3.5:9b


   cargó en 18 s · respuesta 10.2 s (8.1 tokens/s, prompt 1429 tokens) · herramientas ['buscar_productos', 'fijar_presupuesto']
   VRAM tras LLM qwen3.5:9b: 8.70 GB (pico 9.00 GB de 22.0)


,carga,herramientas,carga_s,respuesta_s
modelo,,,,
qwen3.5:9b,✅,✅,18,10.2



Modelo para la prueba punta a punta: qwen3.5:9b


## 5. ASR (faster-whisper) y TTS (XTTS-v2) en GPU

In [7]:
import os

from IPython.display import Audio, display

from asistente_compras.voice import SpeakerSimilarity, XTTSVoice, make_asr, wer

os.environ["COQUI_TOS_AGREED"] = "1"  # licencia CPML (uso no comercial, académico)
REF = paths.root / "voz" / "referencia.wav"
OUT = paths.root / "prueba_colab"

t = time.perf_counter()
asr = make_asr()
RESUMEN["carga ASR (s)"] = round(time.perf_counter() - t, 1)
v1 = next(s for s in VOICE if s.id == "V1")
wav1 = paths.root / "voz" / "escenarios" / "V1" / "turno_01.wav"
tr = asr.transcribe(wav1)
print(f"ASR ({asr.name}): «{tr.text}» · {tr.seconds:.2f} s · WER {wer(v1.turns[0], tr.text):.3f}")
vram("ASR", RESUMEN)

t = time.perf_counter()
tts = XTTSVoice(REF)
RESUMEN["carga TTS (s)"] = round(time.perf_counter() - t, 1)
syn = tts.synthesize("Tengo tres opciones de cables USB tipo C. La primera cuesta 8,99 €. "
                     "¿Cuál prefieres?", OUT / "tts_prueba.wav")
print(f"TTS ({tts.name}): síntesis {syn['seconds']:.1f} s para {syn['audio_seconds']:.1f} s de audio "
      f"(RTF {syn['seconds'] / syn['audio_seconds']:.2f}; en la Mac con CPU ~1,4)")
sim = SpeakerSimilarity(device="cpu")  # usar solo para evaluar: no ocupa la GPU
print(f"Semejanza con la referencia: {sim.similarity(REF, syn['path']):.3f} (en la Mac ~0,81)")
vram("TTS + semejanza", RESUMEN)
display(Audio(str(syn["path"])))

ASR (faster-whisper:large-v3-turbo): «Hola, necesito dos cables USB tipo C. Mi presupuesto es de 40 euros.» · 0.70 s · WER 0.000
   VRAM tras ASR: 8.70 GB (pico 9.00 GB de 22.0)


TTS (xtts_v2@cuda): síntesis 3.9 s para 8.1 s de audio (RTF 0.48; en la Mac con CPU ~1,4)


Semejanza con la referencia: 0.813 (en la Mac ~0,81)
   VRAM tras TTS + semejanza: 10.44 GB (pico 10.60 GB de 22.0)


## 6. Punta a punta: escenario de voz V1

In [8]:
from asistente_compras.agent import format_turn
from asistente_compras.scenarios import evaluate, run_scenario, save_run

torch.cuda.empty_cache()

audio_v1 = [paths.root / "voz" / "escenarios" / "V1" / f"turno_{i:02d}.wav"
            for i in range(1, len(v1.turns) + 1)]
run = run_scenario(v1, llm, retriever, catalog, paths.root / "store" / "prueba_colab.db",
                   audio=audio_v1, asr=asr, tts=tts, audio_out_dir=OUT / "audio")
ev = evaluate(run)
traza = save_run(run, OUT / "trazas")  # guardar transcripciones, herramientas y estado final (en Drive)
for t, v in zip(run.turns, run.voice, strict=True):  # mostrar qué entendió y qué hizo en cada turno
    print(f"🎙 entendió: «{v['transcript']}»")
    print(format_turn(t))
    print()
print(f"{'✅' if ev['passed'] else '❌'} V1: verificaciones {ev['checks_ok']}/{ev['checks_total']}, "
      f"violaciones {ev['violations']}")
for f in ev["failed_checks"]:
    print("   ↳", f)
lat = pd.DataFrame(run.voice)[["turn", "transcript", "asr_s", "llm_s", "tools_s", "tts_s", "total_s"]]
display(lat.round(2))
RESUMEN["V1 correcto"] = "sí" if ev["passed"] else "no"
RESUMEN["latencia media por turno (s)"] = round(lat.total_s.mean(), 1)
vram("punta a punta", RESUMEN)
display(Audio(run.voice[-1]["audio_out"]))

   ⏱ 13.3 s · LLM [6.0, 6.9] s · herramientas 0.4 s
   ⏱ 8.1 s · LLM [4.1, 3.9] s · herramientas 0.0 s
   ⏱ 8.3 s · LLM [2.1, 2.2, 4.0] s · herramientas 0.0 s
✅ V1: verificaciones 4/4, violaciones 0


,turn,asr_s,llm_s,tools_s,tts_s,total_s
0,1,0.24,12.91,0.42,16.06,29.65
1,2,0.20,8.02,0.03,6.66,14.95
2,3,0.19,8.30,0.03,8.40,16.93


   VRAM tras punta a punta: 10.44 GB (pico 10.89 GB de 22.0)


## 7. Resumen

In [9]:
pico = torch.cuda.max_memory_allocated() / 2**30
total = gpu.total_memory / 2**30
RESUMEN["modelo LLM"] = MODELO
RESUMEN["VRAM pico (GB)"] = round(pico, 2)
RESUMEN["VRAM total (GB)"] = round(total, 1)
display(pd.Series(RESUMEN, name="valor").to_frame())
print(f"Traza completa de V1: {traza}")
print("✅ Cabe todo junto en la GPU." if pico < 0.9 * total else
      "⚠ Muy justo de memoria: cargar TTS solo en el notebook 06 o usar una GPU mayor (L4/A100).")
print(f"→ Siguiente: notebooks 03, 05 (compara {CANDIDATOS}) y 06 con MODELO = '{MODELO}'.")

,valor
carga recuperador (s),31.8
recuperador,3.11
kernels atención lineal,sí
LLM qwen3.5:9b: s por llamada,10.2
LLM qwen3.5:9b,9.0
carga ASR (s),3.6
ASR,9.0
carga TTS (s),20.3
TTS + semejanza,10.6
V1 correcto,sí


✅ Cabe todo junto en la GPU.
→ Siguiente: notebooks 03, 05 (compara ['qwen3.5:9b', 'qwen3:8b', 'qwen2.5:7b']) y 06 con MODELO = 'qwen3.5:9b'.
